**.join use for concatenation of string and " " before it add a space between two strings.**

In [74]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
load_dotenv()

True

In [75]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [76]:
embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1813.92it/s]


In [77]:
video_id = "MY5SatbZMAo"

api = YouTubeTranscriptApi()

try:
    transcript = api.fetch(video_id, languages=["en"])

    text = " ".join(item.text for item in transcript)


except Exception as e:
    print(e)

In [78]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=300)

chunks = splitter.create_documents([text])

In [79]:
chunks[0].page_content

'Translator: Riaki Poništ\nReviewer: Peter van de Ven Thank you so much. I am a journalist. My job is to talk to people\nfrom all walks of life, all over the world. Today, I want to tell you why I decided to do this with my life\nand what I\'ve learned. My story begins in Caracas, Venezuela, in South America, where I grew up; a place that to me was,\nand always will be, filled with magic and wonder. Frоm a very young age, my parents wanted me\nto have a wider view of the world. I remember one time\nwhen I was around seven years old, my dad came up to me and said, "Mariana, I\'m going to send you\nand your little sister..." - who was six at the time - "...to a place where nobody\nspeaks Spanish. I want you to experience\ndifferent cultures." He went on and on about the benefits\nof spending an entire summer in this summer camp in the United States, stressing a little phrase that I didn\'t pay'

In [80]:
vectors = FAISS.from_documents(chunks,embeddings)

In [81]:
vectors.index.ntotal

21

In [82]:
retriever = vectors.as_retriever(search_type="similarity",kwargs={"k":4})

In [83]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer from the below context. If the answer is not in the context,
say "I don't have enough information."

Context:
{context}

Question:
{question}


""")

In [87]:
question = "What do you make special?"
retrieved_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [88]:
final_prompt = prompt.invoke({"context":context,"question":question})

In [89]:
answer = model.invoke(final_prompt)
print(answer.content)

The speaker's dance class stood out and made them feel special. The context also states that imperfections, being different, quirky, and unique make us wonderfully human and special.
